# Notebook 06: SQL Analysis
## SaaS Expansion Analytics — RavenStack Dataset
**Objective:** Replicate key expansion metrics using SQL on DuckDB.
Demonstrates multi-table joins across all 5 tables — the same
analysis as Notebooks 02 and 03 but in pure SQL.

In [1]:
import duckdb
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print("Libraries loaded")

Libraries loaded


In [2]:
# Load all 5 CSVs into in-memory DuckDB
conn = duckdb.connect()

accounts      = pd.read_csv('../data/raw/ravenstack_accounts.csv')
subscriptions = pd.read_csv('../data/raw/ravenstack_subscriptions.csv')
feature_usage = pd.read_csv('../data/raw/ravenstack_feature_usage.csv')
support       = pd.read_csv('../data/raw/ravenstack_support_tickets.csv')
churn         = pd.read_csv('../data/raw/ravenstack_churn_events.csv')

conn.register('accounts',      accounts)
conn.register('subscriptions', subscriptions)
conn.register('feature_usage', feature_usage)
conn.register('support',       support)
conn.register('churn',         churn)

print("Tables registered in DuckDB:")
print(conn.execute("SHOW TABLES").fetchdf())

Tables registered in DuckDB:
            name
0       accounts
1          churn
2  feature_usage
3  subscriptions
4        support


## Query 1: MRR by Plan Tier and Month
Monthly recurring revenue breakdown — how does each plan
contribute to total MRR over time?

In [7]:
q1 = conn.execute("""
    SELECT
        STRFTIME(start_date::DATE, '%Y-%m')         AS month,
        plan_tier,
        COUNT(*)                                     AS subscriptions,
        SUM(mrr_amount)                              AS total_mrr,
        ROUND(AVG(mrr_amount), 0)                    AS avg_mrr,
        ROUND(SUM(mrr_amount) * 100.0
            / SUM(SUM(mrr_amount)) OVER
            (PARTITION BY STRFTIME(start_date::DATE,
            '%Y-%m')), 1)                            AS pct_of_month_mrr
    FROM subscriptions
    WHERE mrr_amount > 0
      AND end_date IS NULL
    GROUP BY month, plan_tier
    ORDER BY month, plan_tier
""").fetchdf()

print("=== MRR by Plan Tier and Month (sample) ===")
print(q1.head(12).to_string(index=False))

=== MRR by Plan Tier and Month (sample) ===
  month  plan_tier  subscriptions  total_mrr  avg_mrr  pct_of_month_mrr
2023-01      Basic              1     171.00   171.00              3.70
2023-01 Enterprise              1    3582.00  3582.00             76.50
2023-01        Pro              1     931.00   931.00             19.90
2023-02      Basic              1      76.00    76.00              1.00
2023-02 Enterprise              2    4776.00  2388.00             61.30
2023-02        Pro              3    2940.00   980.00             37.70
2023-03      Basic              3    1862.00   621.00              7.80
2023-03 Enterprise              5   17114.00  3423.00             72.10
2023-03        Pro              4    4753.00  1188.00             20.00
2023-04      Basic              6    4332.00   722.00             10.40
2023-04 Enterprise              8   24079.00  3010.00             58.00
2023-04        Pro              9   13132.00  1459.00             31.60


In [8]:
fig = go.Figure()

for plan in ['Basic', 'Pro', 'Enterprise']:
    df_plan = q1[q1['plan_tier'] == plan]
    fig.add_trace(go.Scatter(
        x=df_plan['month'],
        y=df_plan['total_mrr'],
        name=plan,
        mode='lines+markers'
    ))

fig.update_layout(
    title='MRR by Plan Tier Over Time',
    xaxis_title='Month',
    yaxis_title='MRR ($)',
    height=400,
    font=dict(size=13)
)
fig.show()

## Query 2: Upgrade Rates by Industry and Referral Source
Which segments produce the most expansion?
Joins subscriptions → accounts for enrichment.

In [9]:
q2 = conn.execute("""
    SELECT
        a.industry,
        a.referral_source,
        COUNT(DISTINCT a.account_id)                AS total_accounts,
        COUNT(s.subscription_id)                     AS total_subscriptions,
        SUM(CASE WHEN s.upgrade_flag = TRUE
            THEN 1 ELSE 0 END)                       AS upgrades,
        ROUND(SUM(CASE WHEN s.upgrade_flag = TRUE
            THEN 1.0 ELSE 0 END)
            / NULLIF(COUNT(s.subscription_id),
            0) * 100, 1)                             AS upgrade_rate_pct,
        ROUND(AVG(s.mrr_amount), 0)                  AS avg_mrr
    FROM accounts a
    JOIN subscriptions s
        ON a.account_id = s.account_id
    WHERE s.mrr_amount > 0
    GROUP BY a.industry, a.referral_source
    ORDER BY upgrade_rate_pct DESC
""").fetchdf()

print("=== Upgrade Rates by Industry and Referral Source ===")
print(q2.to_string(index=False))

=== Upgrade Rates by Industry and Referral Source ===
     industry referral_source  total_accounts  total_subscriptions  upgrades  upgrade_rate_pct  avg_mrr
   HealthTech           event              16                  126     20.00             15.90  2894.00
   HealthTech           other              17                  137     20.00             14.60  2484.00
      FinTech         partner              19                  150     21.00             14.00  2407.00
     DevTools             ads              26                  203     28.00             13.80  2100.00
   HealthTech             ads              18                  143     19.00             13.30  2313.00
   HealthTech         partner              14                  121     16.00             13.20  2594.00
      FinTech         organic              20                  171     22.00             12.90  2786.00
Cybersecurity         partner              20                  188     24.00             12.80  2650.00
Cybersecur

## Query 3: Account-Level Feature Summary
Joins all 5 tables into one account-level view.
This is the SQL equivalent of the feature engineering in Notebook 03.

In [10]:
q3 = conn.execute("""
    WITH sub_summary AS (
        SELECT
            account_id,
            COUNT(subscription_id)              AS n_subscriptions,
            SUM(mrr_amount)                     AS total_mrr,
            SUM(seats)                          AS total_seats,
            MAX(CASE WHEN upgrade_flag = TRUE
                THEN 1 ELSE 0 END)              AS has_upgraded,
            MAX(CASE WHEN downgrade_flag = TRUE
                THEN 1 ELSE 0 END)              AS has_downgraded
        FROM subscriptions
        WHERE mrr_amount > 0
          AND end_date IS NULL
        GROUP BY account_id
    ),
    usage_summary AS (
        SELECT
            a.account_id,
            COUNT(fu.usage_id)                  AS total_usage_events,
            SUM(fu.usage_count)                 AS total_usage_count,
            SUM(fu.usage_duration_secs)         AS total_usage_duration,
            COUNT(DISTINCT fu.feature_name)     AS unique_features_used,
            SUM(fu.error_count)                 AS total_errors
        FROM subscriptions s
        JOIN feature_usage fu
            ON s.subscription_id = fu.subscription_id
        JOIN accounts a
            ON s.account_id = a.account_id
        WHERE s.end_date IS NULL
        GROUP BY a.account_id
    ),
    support_summary AS (
        SELECT
            account_id,
            COUNT(ticket_id)                    AS n_tickets,
            ROUND(AVG(satisfaction_score), 2)   AS avg_satisfaction,
            SUM(CASE WHEN escalation_flag = TRUE
                THEN 1 ELSE 0 END)              AS n_escalations,
            ROUND(AVG(resolution_time_hours),
                2)                              AS avg_resolution_hours
        FROM support
        GROUP BY account_id
    )
    SELECT
        a.account_id,
        a.account_name,
        a.industry,
        a.plan_tier,
        a.referral_source,
        a.churn_flag,
        ss.n_subscriptions,
        ss.total_mrr,
        ss.total_seats,
        ss.has_upgraded,
        ss.has_downgraded,
        COALESCE(us.total_usage_count, 0)       AS total_usage_count,
        COALESCE(us.total_usage_duration, 0)    AS total_usage_duration,
        COALESCE(us.unique_features_used, 0)    AS unique_features_used,
        COALESCE(us.total_errors, 0)            AS total_errors,
        COALESCE(sp.n_tickets, 0)               AS n_tickets,
        COALESCE(sp.avg_satisfaction, 0)        AS avg_satisfaction,
        COALESCE(sp.n_escalations, 0)           AS n_escalations
    FROM accounts a
    LEFT JOIN sub_summary ss
        ON a.account_id = ss.account_id
    LEFT JOIN usage_summary us
        ON a.account_id = us.account_id
    LEFT JOIN support_summary sp
        ON a.account_id = sp.account_id
    WHERE a.churn_flag = FALSE
    ORDER BY ss.total_mrr DESC NULLS LAST
""").fetchdf()

print(f"Account feature summary: {q3.shape}")
print(f"\nTop 10 accounts by MRR:")
print(q3.head(10)[[
    'account_name', 'industry', 'plan_tier',
    'total_mrr', 'total_seats', 'total_usage_count',
    'unique_features_used', 'has_upgraded'
]].to_string(index=False))

Account feature summary: (390, 18)

Top 10 accounts by MRR:
account_name industry  plan_tier  total_mrr  total_seats  total_usage_count  unique_features_used  has_upgraded
 Company_166 DevTools        Pro  131911.00       949.00             793.00                    34             1
 Company_403  FinTech      Basic  114777.00      1053.00             459.00                    30             1
 Company_368   EdTech        Pro   94710.00       840.00             625.00                    29             1
 Company_130   EdTech      Basic   83886.00       744.00             475.00                    27             0
 Company_488   EdTech      Basic   75776.00      1184.00             419.00                    28             0
 Company_480   EdTech Enterprise   70980.00       420.00             363.00                    24             1
 Company_341  FinTech Enterprise   70000.00       700.00             566.00                    28             1
 Company_358   EdTech Enterprise   69687.00 

## Query 4: NRR Calculation
Net Revenue Retention — the single most important SaaS health metric.
Calculated entirely in SQL across subscriptions table.

In [11]:
q4 = conn.execute("""
    WITH mrr_components AS (
        SELECT
            SUM(mrr_amount)                         AS total_mrr,
            SUM(CASE WHEN upgrade_flag = TRUE
                THEN mrr_amount ELSE 0 END)         AS expansion_mrr,
            SUM(CASE WHEN churn_flag = TRUE
                THEN mrr_amount ELSE 0 END)         AS churned_mrr,
            SUM(CASE WHEN downgrade_flag = TRUE
                THEN mrr_amount ELSE 0 END)         AS contraction_mrr
        FROM subscriptions
        WHERE mrr_amount > 0
    )
    SELECT
        total_mrr                                   AS starting_mrr,
        expansion_mrr,
        churned_mrr,
        contraction_mrr,
        total_mrr
            + expansion_mrr
            - churned_mrr
            - contraction_mrr                       AS ending_mrr,
        ROUND((total_mrr
            + expansion_mrr
            - churned_mrr
            - contraction_mrr)
            * 100.0 / total_mrr, 1)                 AS nrr_pct
    FROM mrr_components
""").fetchdf()

print("=== Net Revenue Retention ===")
print(f"Starting MRR:    ${q4['starting_mrr'].iloc[0]:>12,.0f}")
print(f"+ Expansion MRR: ${q4['expansion_mrr'].iloc[0]:>12,.0f}")
print(f"- Churned MRR:   ${q4['churned_mrr'].iloc[0]:>12,.0f}")
print(f"- Contraction:   ${q4['contraction_mrr'].iloc[0]:>12,.0f}")
print(f"{'─' * 40}")
print(f"Ending MRR:      ${q4['ending_mrr'].iloc[0]:>12,.0f}")
print(f"\nNRR:             {q4['nrr_pct'].iloc[0]}%")

=== Net Revenue Retention ===
Starting MRR:    $  11,338,747
+ Expansion MRR: $   1,262,997
- Churned MRR:   $   1,179,139
- Contraction:   $     459,366
────────────────────────────────────────
Ending MRR:      $  10,963,239

NRR:             96.7%


## Query 5: Top Expansion Targets
Joins all signal tables to rank active Basic and Pro accounts
by expansion potential — the SQL version of our scored target list.

In [12]:
q5 = conn.execute("""
    WITH sub_signals AS (
        SELECT
            account_id,
            SUM(mrr_amount)                         AS total_mrr,
            SUM(seats)                              AS total_seats,
            COUNT(subscription_id)                  AS n_subscriptions
        FROM subscriptions
        WHERE mrr_amount > 0
          AND end_date IS NULL
        GROUP BY account_id
    ),
    usage_signals AS (
        SELECT
            a.account_id,
            SUM(fu.usage_duration_secs)             AS total_usage_duration,
            COUNT(DISTINCT fu.feature_name)         AS unique_features_used
        FROM subscriptions s
        JOIN feature_usage fu
            ON s.subscription_id = fu.subscription_id
        JOIN accounts a
            ON s.account_id = a.account_id
        WHERE s.end_date IS NULL
        GROUP BY a.account_id
    ),
    support_signals AS (
        SELECT
            account_id,
            ROUND(AVG(satisfaction_score), 2)       AS avg_satisfaction,
            SUM(CASE WHEN escalation_flag = TRUE
                THEN 1 ELSE 0 END)                  AS n_escalations
        FROM support
        GROUP BY account_id
    )
    SELECT
        a.account_id,
        a.account_name,
        a.industry,
        a.plan_tier,
        a.referral_source,
        ss.total_mrr,
        ss.total_seats,
        ss.n_subscriptions,
        COALESCE(us.total_usage_duration, 0)        AS total_usage_duration,
        COALESCE(us.unique_features_used, 0)        AS unique_features_used,
        COALESCE(sp.avg_satisfaction, 0)            AS avg_satisfaction,
        COALESCE(sp.n_escalations, 0)               AS n_escalations,
        -- Composite expansion signal score
        ROUND(
            (COALESCE(us.total_usage_duration, 0)
                / 100000.0 * 40)
            + (ss.total_seats / 500.0 * 30)
            + (ss.total_mrr / 50000.0 * 20)
            + (ss.n_subscriptions / 20.0 * 10)
        , 1)                                        AS sql_expansion_score
    FROM accounts a
    JOIN sub_signals ss
        ON a.account_id = ss.account_id
    LEFT JOIN usage_signals us
        ON a.account_id = us.account_id
    LEFT JOIN support_signals sp
        ON a.account_id = sp.account_id
    WHERE a.churn_flag = FALSE
      AND a.plan_tier IN ('Basic', 'Pro')
    ORDER BY sql_expansion_score DESC
    LIMIT 20
""").fetchdf()

print("=== Top 20 Expansion Targets (SQL Score) ===")
print(q5[[
    'account_name', 'industry', 'plan_tier',
    'total_mrr', 'total_seats',
    'unique_features_used', 'sql_expansion_score'
]].to_string(index=False))

=== Top 20 Expansion Targets (SQL Score) ===
account_name      industry plan_tier  total_mrr  total_seats  unique_features_used  sql_expansion_score
 Company_166      DevTools       Pro  131911.00       949.00                    34               207.40
 Company_235        EdTech     Basic   64400.00       920.00                    32               170.10
 Company_403       FinTech     Basic  114777.00      1053.00                    30               167.30
 Company_368        EdTech       Pro   94710.00       840.00                    29               166.60
 Company_488        EdTech     Basic   75776.00      1184.00                    28               163.20
 Company_174       FinTech     Basic   69230.00       770.00                    31               160.20
 Company_430      DevTools     Basic   42740.00       320.00                    34               147.10
  Company_40       FinTech     Basic   51109.00       541.00                    32               141.80
  Company_82      D

## Key Findings

**Query 1 — Enterprise dominates MRR from day one:**
Enterprise contributed 76.5% of MRR in the first month
despite only one subscription. The revenue concentration
in Enterprise is structural — Basic and Pro are the
feeding pipeline not the revenue base.

**Query 2 — Industry matters more than referral source:**
HealthTech upgrades at 15.9% (event channel) — highest
of any segment combination. EdTech upgrades at just 5.6%
(ads channel) — lowest. HealthTech outperforms EdTech
regardless of referral source, suggesting industry
fit drives expansion more than acquisition channel.

**Query 3 — Five-table join produces 390 account profiles:**
Multi-table CTE joins accounts, subscriptions,
feature_usage and support into a single account
intelligence view — the SQL equivalent of the
pandas feature engineering in Notebook 03.
Three high-MRR Basic accounts (Company_130 $83K,
Company_488 $75K, Company_174 $69K) have not upgraded
— strong expansion targets not surfaced by the ML model.

**Query 4 — NRR confirmed at 96.7%:**
SQL calculation produces identical results to pandas
in Notebook 02. Cross-validation between SQL and Python
confirms the NRR methodology is correct.
Contraction MRR ($459K) remains the primary drag.

**Query 5 — SQL and ML scores surface different targets:**
SQL score weights raw size (MRR 20%, seats 30%) heavily —
large accounts dominate the top 20.
ML score weights behavioural patterns — usage duration,
subscription history, engagement signals.
Five accounts appear in both top 20 lists:
Company_40, Company_82, Company_258, Company_287, Company_364
— these are the highest confidence expansion targets,
validated by two independent methodologies.